## IMDB_sentiment_Analysis_project

In [5]:
# # NLP Project Lifecycle :
# >> Data Gathering
# >> EDA & Text Preprocessing
# >> Text representation
# >> Model Building
# >> Model Evaluation
# >> Model Deployment

# ## It is not linear but iterative process

# Data Gathering :

In [6]:
import pandas as pd

train_df = pd.read_csv("/content/IMDB-Train.csv")
train_df

,review,sentiment
0,That's what I kept asking myself during the ma...,negative
1,I did not watch the entire movie. I could not ...,negative
2,A touching love story reminiscent of In the M...,positive
3,This latter-day Fulci schlocker is a totally a...,negative
4,"First of all, I firmly believe that Norwegian ...",negative
...,...,...
39995,`Shadow Magic' recaptures the joy and amazemen...,positive
39996,I found this movie to be quite enjoyable and f...,positive
39997,Avoid this one! It is a terrible movie. So wha...,negative
39998,This production was quite a surprise for me. I...,positive


In [7]:
test_df = pd.read_csv("/content/IMDB-Test.csv")
test_df

,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative
...,...,...
9995,Although Casper van Dien and Michael Rooker ar...,negative
9996,I liked this movie. I wasn't really sure what ...,positive
9997,Yes non-Singaporean's can't see what's the big...,positive
9998,"As far as films go, this is likable enough. En...",negative


## Cleaning of both train and test dataset

In [8]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
english_stops = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
def load_dataset(df):

    x_train = df['review']       # Reviews/Input
    y_train = df['sentiment']    # Sentiment/Output

    # PRE-PROCESS REVIEW
    x_train = x_train.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_train = x_train.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_train = x_train.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
    x_train = x_train.apply(lambda review: [w.lower() for w in review])   # lower case

    # ENCODE SENTIMENT -> 0 & 1
    y_train = y_train.replace('positive', 1)
    y_train = y_train.replace('negative', 0)

    return x_train, y_train



In [10]:
x_train, y_train = load_dataset(train_df)

print('Reviews')
print(x_train, '\n')
print('Sentiment')
print(y_train)

Reviews
0        [that, i, kept, asking, many, fights, screamin...
1        [i, watch, entire, movie, i, could, watch, ent...
2        [a, touching, love, story, reminiscent, in, mo...
3        [this, latter, day, fulci, schlocker, totally,...
4        [first, i, firmly, believe, norwegian, movies,...
                               ...                        
39995    [shadow, magic, recaptures, joy, amazement, fi...
39996    [i, found, movie, quite, enjoyable, fairly, en...
39997    [avoid, one, it, terrible, movie, so, exciting...
39998    [this, production, quite, surprise, i, absolut...
39999    [this, decent, movie, although, little, bit, s...
Name: review, Length: 40000, dtype: object 

Sentiment
0        0
1        0
2        1
3        0
4        0
        ..
39995    1
39996    1
39997    0
39998    1
39999    1
Name: sentiment, Length: 40000, dtype: int64


/tmp/ipykernel_601/1908888035.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace('negative', 0)


In [11]:
print(y_train.value_counts())

sentiment
0    20039
1    19961
Name: count, dtype: int64


In [12]:
x_test, y_test = load_dataset(test_df)

print('Reviews')
print(x_test, '\n')
print('Sentiment')
print(y_test)

Reviews
0       [i, really, liked, summerslam, due, look, aren...
1       [not, many, television, shows, appeal, quite, ...
2       [the, film, quickly, gets, major, chase, scene...
3       [jane, austen, would, definitely, approve, one...
4       [expectations, somewhat, high, i, went, see, m...
                              ...                        
9995    [although, casper, van, dien, michael, rooker,...
9996    [i, liked, movie, i, really, sure, i, started,...
9997    [yes, non, singaporean, see, big, deal, film, ...
9998    [as, far, films, go, likable, enough, entertai...
9999    [i, saw, anatomy, years, ago, dubbed, friends,...
Name: review, Length: 10000, dtype: object 

Sentiment
0       1
1       1
2       0
3       1
4       0
       ..
9995    0
9996    1
9997    1
9998    0
9999    1
Name: sentiment, Length: 10000, dtype: int64


/tmp/ipykernel_601/1908888035.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace('negative', 0)


# Tokenize and Padding Reviews

In [13]:
# Function for getting the maximum review length, by calculating the mean of all the reviews length (using numpy.mean)
import numpy as np

def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))

    return int(np.ceil(np.mean(review_length)))

# ENCODE REVIEW

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating

token = Tokenizer(lower=False)    # no need lower, because already lowered the data in load_data()
token.fit_on_texts(x_train)
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1   # add 1 because of 0 padding

print('Encoded X Train\n', x_train, '\n\n')
print('Encoded X Test\n', x_test, '\n\n')
print('Maximum review length: ', max_length)

Encoded X Train
 [[  145     1   702 ...   903 16627   401]
 [    1    33   347 ...    31     3   545]
 [   39  1243    42 ...     0     0     0]
 ...
 [  694     5     7 ...     0     0     0]
 [    8   259    89 ...     0     0     0]
 [    8   442     3 ...     0     0     0]] 


Encoded X Test
 [[    1    14   329 ...     0     0     0]
 [  155    38   601 ...   849  2484   260]
 [    2     4   832 ...     0     0     0]
 ...
 [  333   597 24772 ...     0     0     0]
 [  108   131    35 ...     0     0     0]
 [    1   120  8036 ...     0     0     0]] 


Maximum review length:  130


In [15]:
# ==== NEW CELL: Save Tokenizer + max_length + stopwords ====
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(token, f)

with open("max_length.pkl", "wb") as f:
    pickle.dump(max_length, f)

with open("english_stops.pkl", "wb") as f:
    pickle.dump(english_stops, f)

print("Saved tokenizer.pkl, max_length.pkl, english_stops.pkl")

Saved tokenizer.pkl, max_length.pkl, english_stops.pkl


# Model Building

In [16]:
# ARCHITECTURE

from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense # layers of the architecture


EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

model.build(input_shape=(None, max_length))
print(model.summary())




/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 130, 32)        │     2,961,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,986,369 (11.39 MB)

 Trainable params: 2,986,369 (11.39 MB)

 Non-trainable params: 0 (0.00 B)

None


# Training

In [17]:
# from tensorflow.keras.callbacks import ModelCheckpoint   # save model

# checkpoint = ModelCheckpoint(
#     'models/LSTM.h5',
#     monitor='accuracy',
#     save_best_only=True,
#     verbose=1
# )

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    # filepath="best_lstm_model.keras",
    filepath="best_lstm_model.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# early_stop = EarlyStopping(
#     monitor="val_accuracy",
#     mode="max",
#     patience=3,
#     restore_best_weights=True,
#     verbose=1
# )

In [18]:
# history = model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])

# print(history.history)


history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=128,
    callbacks=[checkpoint]
    # callbacks=[checkpoint, early_stop]
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.5333 - loss: 0.6803
Epoch 1: val_accuracy improved from None to 0.59390, saving model to best_lstm_model.h5



Epoch 1: finished saving model to best_lstm_model.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 73s 227ms/step - accuracy: 0.5797 - loss: 0.6593 - val_accuracy: 0.5939 - val_loss: 0.6445
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.5644 - loss: 0.6612
Epoch 2: val_accuracy did not improve from 0.59390
313/313 ━━━━━━━━━━━━━━━━━━━━ 80s 221ms/step - accuracy: 0.5596 - loss: 0.6759 - val_accuracy: 0.5461 - val_loss: 0.6821
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.6383 - loss: 0.6351
Epoch 3: val_accuracy did not improve from 0.59390
313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 220ms/step - accuracy: 0.6211 - loss: 0.6414 - val_accuracy: 0.5285 - val_loss: 0.6863
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.6161 - loss: 0.6390
Epoch 4: val_accuracy improved from 0.59390 to 0.79680, saving model to best_lstm_model.h5



Epoch 4: finished saving model to best_lstm_model.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 219ms/step - accuracy: 0.6932 - loss: 0.5777 - val_accuracy: 0.7968 - val_loss: 0.4959
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.7841 - loss: 0.4916
Epoch 5: val_accuracy did not improve from 0.79680
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 220ms/step - accuracy: 0.7210 - loss: 0.5463 - val_accuracy: 0.6652 - val_loss: 0.6110
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.8641 - loss: 0.3814
Epoch 6: val_accuracy improved from 0.79680 to 0.83680, saving model to best_lstm_model.h5



Epoch 6: finished saving model to best_lstm_model.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 73s 232ms/step - accuracy: 0.8697 - loss: 0.3668 - val_accuracy: 0.8368 - val_loss: 0.4244
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.8510 - loss: 0.3980
Epoch 7: val_accuracy did not improve from 0.83680
313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 218ms/step - accuracy: 0.8241 - loss: 0.4354 - val_accuracy: 0.8266 - val_loss: 0.4356
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.8888 - loss: 0.2962
Epoch 8: val_accuracy did not improve from 0.83680
313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 220ms/step - accuracy: 0.8568 - loss: 0.3619 - val_accuracy: 0.7956 - val_loss: 0.4773
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.8670 - loss: 0.3517
Epoch 9: val_accuracy did not improve from 0.83680
313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 221ms/step - accuracy: 0.8873 - loss: 0.3110 - val_accuracy: 0.8281 - val_loss: 0.4561
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms

In [20]:
from google.colab import files

files.download("best_lstm_model.h5")

files.download("tokenizer.pkl")
files.download("max_length.pkl")
files.download("english_stops.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Testing

In [21]:

y_prob = model.predict(x_test, batch_size = 128)

y_pred = (y_prob > 0.5).astype(int)

true = 0
for i, y in enumerate(y_test):
    if y == y_pred[i]:
        true += 1

print('Correct Prediction: {}'.format(true))
print('Wrong Prediction: {}'.format(len(y_pred) - true))
print('Accuracy: {}'.format(true/len(y_pred)*100))

79/79 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step
Correct Prediction: 8346
Wrong Prediction: 1654
Accuracy: 83.46000000000001


# Load Saved Model

In [22]:
from tensorflow.keras.models import load_model   # load saved model

loaded_model = load_model('/content/best_lstm_model.h5')

In [28]:
review = str(input('Movie Review: '))

Movie Review: I fully expected to hate this cheap-looking comedy, but the hilarious dialogue actually kept me laughing until the very end.


In [29]:
import re


# Pre-process input
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned: ', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]

print('Filtered: ', filtered)

Cleaned:  I fully expected to hate this cheaplooking comedy but the hilarious dialogue actually kept me laughing until the very end
Filtered:  ['i fully expected hate cheaplooking comedy hilarious dialogue actually kept laughing end']


In [30]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[    1  1301   774   637 68442   116   510   317    74   702   932    54
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0]]


In [31]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[[0.85829353]]


In [32]:
if result[0][0] >= 0.5:
    print("Positive")
else:
    print("Negative")

Positive
